In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import create_spark, project_root
from pyspark.sql import functions as F

spark = create_spark("incremental-integration")
ROOT = project_root()

from src.lake import GOLD, SILVER, read_delta, write_gold

:: loading settings :: url = jar:file:/Users/samuelflodin/programmering/uppgifter/5an/bigdata/id2221-labs/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/samuelflodin/.ivy2/cache
The jars for the packages stored in: /Users/samuelflodin/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4f6a7668-d8ee-4ba2-ae68-68d2132ad381;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 74ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |

## Read updated Silver tables

Silver now contains incremental records merged in by `generated_data_ingestion.ipynb`, including the new `humidity` (weather) and `aqi` (air quality) columns.

In [2]:
trips = read_delta(spark, SILVER / "taxi_trips")
weather = read_delta(spark, SILVER / "weather")
air_quality = read_delta(spark, SILVER / "air_quality")
zones = read_delta(spark, SILVER / "taxi_zones")

print("Schema-evolved columns present:")
print("  humidity:", "humidity" in weather.columns)
print("  aqi:     ", "aqi" in air_quality.columns)

Schema-evolved columns present:
  humidity: True
  aqi:      True


## Prepare dimension tables

In [3]:
weather_agg_cols = [
    F.avg("temperature_c").alias("temperature_c"),
    F.avg("wind_speed_ms").alias("wind_speed_ms"),
]
if "humidity" in weather.columns:
    weather_agg_cols.append(F.avg("humidity").alias("humidity"))

hourly_weather = (
    weather
    .groupBy(
        F.col("observation_date").alias("pickup_date"),
        F.col("observation_hour").alias("pickup_hour"),
    )
    .agg(*weather_agg_cols)
)

print(f"hourly_weather: {hourly_weather.count():,} hours")
hourly_weather.show(3, truncate=False)

26/09/24 16:20:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


hourly_weather: 8,872 hours
+-----------+-----------+-------------+-----------------+--------+
|pickup_date|pickup_hour|temperature_c|wind_speed_ms    |humidity|
+-----------+-----------+-------------+-----------------+--------+
|2025-01-03 |0          |12.8         |9.305555555555555|86.0    |
|2025-01-03 |2          |12.2         |4.111111111111112|86.0    |
|2025-01-03 |8          |13.9         |7.694444444444444|49.0    |
+-----------+-----------+-------------+-----------------+--------+
only showing top 3 rows



In [4]:
NYC_COUNTIES = [5, 47, 61, 81, 85]

aq_agg_cols = [
    F.avg("value").alias("pm25"),
    F.first("unit").alias("pm25_unit"),
]
if "aqi" in air_quality.columns:
    aq_agg_cols.append(F.avg("aqi").cast("int").alias("aqi"))

hourly_aq = (
    air_quality
    .filter(F.col("county_code").isin(NYC_COUNTIES))
    .groupBy(
        F.col("measurement_date").alias("pickup_date"),
        F.col("measurement_hour").alias("pickup_hour"),
    )
    .agg(*aq_agg_cols)
)

print(f"hourly_aq: {hourly_aq.count():,} hours")
hourly_aq.show(3, truncate=False)

hourly_aq: 8,871 hours
+-----------+-----------+------------------+---------------------------+----+
|pickup_date|pickup_hour|pm25              |pm25_unit                  |aqi |
+-----------+-----------+------------------+---------------------------+----+
|2024-01-01 |0          |14.520000000000001|Micrograms/cubic meter (LC)|NULL|
|2024-01-01 |1          |14.459999999999999|Micrograms/cubic meter (LC)|NULL|
|2024-01-01 |2          |14.440000000000001|Micrograms/cubic meter (LC)|NULL|
+-----------+-----------+------------------+---------------------------+----+
only showing top 3 rows



In [5]:
pickup_zones = zones.select(
    F.col("location_id").alias("pickup_location_id"),
    F.col("zone").alias("pickup_zone"),
    F.col("borough").alias("pickup_borough"),
)

dropoff_zones = zones.select(
    F.col("location_id").alias("dropoff_location_id"),
    F.col("zone").alias("dropoff_zone"),
    F.col("borough").alias("dropoff_borough"),
)

## Join and write Gold

Uses `SELECT *` on the joined result so new columns (`humidity`, `aqi`) flow through automatically without hardcoding column names.

In [6]:
import time

integrated = (
    trips
    .join(F.broadcast(pickup_zones), "pickup_location_id", "left")
    .join(F.broadcast(dropoff_zones), "dropoff_location_id", "left")
    .join(F.broadcast(hourly_weather), ["pickup_date", "pickup_hour"], "left")
    .join(F.broadcast(hourly_aq), ["pickup_date", "pickup_hour"], "left")
    .fillna(
        {
            "pickup_zone": "UNKNOWN",
            "pickup_borough": "UNKNOWN",
            "dropoff_zone": "UNKNOWN",
            "dropoff_borough": "UNKNOWN",
        }
    )
    .drop("_ingested_at")
    .cache()
)

integrated.count()

t0 = time.perf_counter()
write_gold(integrated, "integrated_taxi_trips", partition_by=["pickup_date"])
print(f"write by_date: {time.perf_counter() - t0:.1f}s")

t0 = time.perf_counter()
write_gold(integrated, "integrated_taxi_trips_by_borough", partition_by=["pickup_borough"])
print(f"write by_borough: {time.perf_counter() - t0:.1f}s")

print("\nFinal schema:")
integrated.printSchema()

write by_date: 26.8s


write by_borough: 37.6s

Final schema:
root
 |-- pickup_date: date (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- dropoff_location_id: integer (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- taxi_type: string (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- pickup_zone: string (nullable = false)
 |-- pickup_borough: string (nullable = false)
 |-- dropoff_zone: string (nullable = false)
 |-- dropoff_borough: string (nullable = false)
 |-- temperature_c: double (nullable = true)
 |-- wind_speed_ms: double (nullable = true)
 |-- humidity: double (nullable = true)
 |--